# **Milestone 1: Smart MCQ Solver Challenge**
Make sure to upload your `train (1).csv` file to the Colab files section before executing the blocks.

In [ ]:
import string
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

# Load Dataset
df = pd.read_csv("train (1).csv").fillna("")
print(f"Dataset loaded successfully with {len(df)} rows.")

In [ ]:
# Helper function and setup
translator = str.maketrans('', '', string.punctuation)

def clean_and_split(text):
    text = str(text).lower()
    text = text.translate(translator)
    return text.split()

option_cols = ["A", "B", "C", "D", "E"]

def map3_score(actual, preds):
    for rank, pred in enumerate(preds):
        if pred == actual:
            return 1 / (rank + 1)
    return 0

### **Question 1**

In [ ]:
counts = df["answer"].value_counts()

print("--- Q1 ---")
print(counts)
print()

most = counts.max()
least = counts.min()
print("Answer =", most + least)

### **Question 2**

In [ ]:
vocab = set()
for prompt in df["prompt"]:
    vocab.update(clean_and_split(prompt))

print("--- Q2 ---")
print("Vocabulary Size =", len(vocab))

### **Question 3**

In [ ]:
if "id" in df.columns:
    row = df[df["id"] == 1].iloc[0]
else:
    row = df.iloc[0]

words = clean_and_split(row["prompt"])
filtered = [w for w in words if w not in ENGLISH_STOP_WORDS]

print("--- Q3 ---")
print("Remaining Words =", len(filtered))

### **Question 4**

In [ ]:
corpus = []
for _, row in df.iterrows():
    corpus.extend([
        row["prompt"],
        row["A"],
        row["B"],
        row["C"],
        row["D"],
        row["E"]
    ])

vectorizer = TfidfVectorizer(stop_words="english")
vectorizer.fit(corpus)

print("--- Q4 ---")
print("Vocabulary Size =", len(vectorizer.vocabulary_))

### **Precompute TF-IDF Vectors**
This steps extracts features from the dataset before applying them to downstream calculations.

In [ ]:
prompt_vectors = vectorizer.transform(df["prompt"])
option_vectors = {
    opt: vectorizer.transform(df[opt])
    for opt in option_cols
}
print("TF-IDF transformations completed.")

### **Question 5**

In [ ]:
score = cosine_similarity(
    prompt_vectors[0],
    option_vectors["A"][0]
)[0][0]

print("--- Q5 ---")
print("Cosine Similarity =", round(score, 4))

### **Question 6**

In [ ]:
correct = 0
for i in range(len(df)):
    sims = []
    for opt in option_cols:
        sim = cosine_similarity(
            prompt_vectors[i],
            option_vectors[opt][i]
)[0][0]
        sims.append((opt, sim))
    
    sims.sort(key=lambda x: (-x[1], x[0]))
    prediction = sims[0][0]
    
    if prediction == df.iloc[i]["answer"]:
        correct += 1

accuracy = correct / len(df) * 100
print("--- Q6 ---")
print(f"Accuracy = {accuracy:.2f}%")

### **Question 7 & Question 8**

In [ ]:
print("--- Q7 ---")
print("MAP@3 for [C, A, B] =", map3_score("C", ["C", "A", "B"]))
print()
print("--- Q8 ---")
print("MAP@3 for [D, B, E] =", map3_score("B", ["D", "B", "E"]))

### **Question 9**

In [ ]:
top3 = counts.index.tolist()[:3]
score = 0

for ans in df["answer"]:
    score += map3_score(ans, top3)

score /= len(df)
print("--- Q9 ---")
print("MAP@3 =", round(score, 4))
print("Top 3 Baseline =", top3)

### **Question 10**

In [ ]:
total = 0
for i in range(len(df)):
    sims = []
    for opt in option_cols:
        sim = cosine_similarity(
            prompt_vectors[i],
            option_vectors[opt][i]
)[0][0]
        sims.append((opt, sim))
    
    sims.sort(key=lambda x: (-x[1], x[0]))
    preds = [x[0] for x in sims[:3]]
    total += map3_score(df.iloc[i]["answer"], preds)

map3 = total / len(df)
print("--- Q10 ---")
print("Average MAP@3 =", round(map3, 4))